In [1]:
import pandas as pd
from scipy import stats

In [2]:
data = pd.read_excel("/Users/esther/PycharmProjects/ad_proteomic_clock/training_data/JPST003472.xlsx").set_index("sample_id")

In [3]:
# Extract timepoint from sample_id
timepoint = pd.Series(data.index.str.split('_').str[0], index=data.index)
print(timepoint.value_counts()) # Check group sizes

baseline = '3M'
comparisons = ['5M', '8M', '14M', '20M', '26M']

baseline_data = data.loc[timepoint[timepoint == baseline].index]
print(f"{baseline}: n={baseline_data.shape[0]}")

dep_results = {}

for t in comparisons:
    t_data = data.loc[timepoint[timepoint == t].index]
    print(f"{t}: n={t_data.shape[0]}")

    p_values = []
    log_fc = []

    # Iterates through all protein columns
    for protein in data.columns:
        stat, p = stats.ttest_ind(t_data[protein], baseline_data[protein], equal_var=False)
        p_values.append(p)
        log_fc.append(t_data[protein].mean() - baseline_data[protein].mean())

    result = pd.DataFrame({
        'protein': data.columns,
        'p_value': p_values,
        'logFC': log_fc
    }).set_index('protein')

    dep_results[t] = result

    n_signf = (result['p_value'] < 0.05).sum()
    print(f"{baseline} vs {t}: {n_signf} proteins with p<0.05 (uncorrected) out of {len(result)}")

# Save to excel
with pd.ExcelWriter("/Users/esther/PycharmProjects/ad_proteomic_clock/results/3M_DEP_results.xlsx") as writer:
    for t in comparisons:
        dep_results[t].sort_values('p_value').to_excel(writer, sheet_name=f"3M_vs_{t}")

sample_id
14M    9
5M     9
8M     8
20M    7
3M     7
26M    5
Name: count, dtype: int64
3M: n=7
5M: n=9
3M vs 5M: 176 proteins with p<0.05 (uncorrected) out of 4122
8M: n=8
3M vs 8M: 221 proteins with p<0.05 (uncorrected) out of 4122
14M: n=9
3M vs 14M: 308 proteins with p<0.05 (uncorrected) out of 4122
20M: n=7
3M vs 20M: 317 proteins with p<0.05 (uncorrected) out of 4122
26M: n=5
3M vs 26M: 202 proteins with p<0.05 (uncorrected) out of 4122


In [4]:
# Get significant protein sets across comparisons
sig_sets = {t: set(dep_results[t][dep_results[t]['p_value'] < 0.05].index) for t in comparisons}

common_deps = set.intersection(*sig_sets.values())
print(f"Proteins significant (p < 0.05) across all {len(comparisons)} comparisons: {len(common_deps)}")

common_deps_df = pd.DataFrame(index=sorted(common_deps))
for t in comparisons:
    common_deps_df[f'pvalue_{t}'] = dep_results[t].loc[common_deps_df.index, 'p_value']
    common_deps_df[f'logFC_{t}'] = dep_results[t].loc[common_deps_df.index, 'logFC']

# Get significant protein sets per comparison
from itertools import combinations

overlap_results = {}

for t1, t2 in combinations(comparisons, 2):
    shared = sig_sets[t1].intersection(sig_sets[t2])
    overlap_results[f'{t1}_and_{t2}'] = sorted(shared)
    print(f"{t1} ∩ {t2}: {len(shared)} shared proteins "
          f"(out of {len(sig_sets[t1])} in {t1}, {len(sig_sets[t2])} in {t2})")

# Save to excel
with pd.ExcelWriter("/Users/esther/PycharmProjects/ad_proteomic_clock/results/3M_DEP_comparisons.xlsx") as writer:
    common_deps_df.to_excel(writer, sheet_name="common_across_3M_comps")

    for key, proteins in overlap_results.items():
        if len(proteins) == 0:
            continue
        t1, t2 = key.split('_and_')
        overlap_df = pd.DataFrame(index=proteins)
        overlap_df[f'pvalue_{t1}'] = dep_results[t1].loc[proteins, 'p_value']
        overlap_df[f'logFC_{t1}'] = dep_results[t1].loc[proteins, 'logFC']
        overlap_df[f'pvalue_{t2}'] = dep_results[t2].loc[proteins, 'p_value']
        overlap_df[f'logFC_{t2}'] = dep_results[t2].loc[proteins, 'logFC']
        overlap_df.to_excel(writer, sheet_name=key[:31])

Proteins significant (p < 0.05) in all 5 comparisons: 5
5M ∩ 8M: 48 shared proteins(out of 176 in 5M, 221 in 8M)
5M ∩ 14M: 46 shared proteins(out of 176 in 5M, 308 in 14M)
5M ∩ 20M: 54 shared proteins(out of 176 in 5M, 317 in 20M)
5M ∩ 26M: 18 shared proteins(out of 176 in 5M, 202 in 26M)
8M ∩ 14M: 87 shared proteins(out of 221 in 8M, 308 in 14M)
8M ∩ 20M: 64 shared proteins(out of 221 in 8M, 317 in 20M)
8M ∩ 26M: 26 shared proteins(out of 221 in 8M, 202 in 26M)
14M ∩ 20M: 114 shared proteins(out of 308 in 14M, 317 in 20M)
14M ∩ 26M: 37 shared proteins(out of 308 in 14M, 202 in 26M)
20M ∩ 26M: 58 shared proteins(out of 317 in 20M, 202 in 26M)
